# Free Throw Model — Cross‑Session Evaluation & Calibration
**Generated:** 2025-08-18 23:21:34

This notebook helps you:
- Run **cross-session** tests (train on one session, test on another).
- Try multiple **rebalancing** strategies (class weights, under/over-sampling, SMOTE).
- Evaluate multiple **models** with **GroupKFold** by session.
- **Calibrate** probabilities and **tune thresholds** for your cost function.
- Produce **confusion matrices**, **PR curves**, **reliability (calibration) curves**, and **CSV reports**.

> **Instructions**
> 1. Edit the paths in the **Config** cell to point to your `X.csv` and `y.csv` for each session.
> 2. Optionally adjust the model list and rebalancing options.
> 3. Run cells top-to-bottom. Outputs will be saved next to each session's dataset under `analysis/datasets/experiments/`.
>
> **Assumptions**
> - `X.csv` is all-numeric feature columns (same schema across sessions).
> - `y.csv` contains a single column `label` with values like `made`/`miss` or `1`/`0` (we map to 1 for made, 0 for miss).
> - Optional column `clip_id` in `X.csv` will be used for error analysis if present.

In [ ]:
# =======================
# Config — EDIT THIS CELL
# =======================

from pathlib import Path

# Example layout (edit to match your repo)
# data/<athlete>/<session>/analysis/datasets/{X.csv,y.csv}
SESSIONS = {
    # session_name : (X_path, y_path, out_root_dir)
    "session_02": (
        Path("data/alton_overson/session_02/analysis/datasets/X.csv"),
        Path("data/alton_overson/session_02/analysis/datasets/y.csv"),
        Path("data/alton_overson/session_02/analysis/datasets"),
    ),
    "session_03": (
        Path("data/alton_overson/session_03/analysis/datasets/X.csv"),
        Path("data/alton_overson/session_03/analysis/datasets/y.csv"),
        Path("data/alton_overson/session_03/analysis/datasets"),
    ),
}

# Experiment folder name (created under each out_root_dir)
EXP_SUBDIR = "experiments"

# Positive class name in your y.csv (e.g., "made" or 1). We coerce to {0,1}.
POSITIVE_CLASS = "made"  # change to 1 if your labels are numeric

# Rebalancing strategy for training folds:
# "none" | "class_weight" | "undersample" | "oversample" | "smote"
REBALANCE = "none"

# Probability calibration
CALIBRATE = True          # wrap model in CalibratedClassifierCV when possible
CALIB_METHOD = "isotonic" # "isotonic" or "sigmoid"

# Threshold selection metric: "f1", "macro_f1", or "custom"
THRESHOLD_OBJECTIVE = "macro_f1"

# If you choose "custom", define the cost weights below (higher = worse):
COST_FN = 2.0  # cost of false negative (miss predicted as make)
COST_FP = 1.0  # cost of false positive (make predicted as miss)

# Models to evaluate.
# You can comment models in/out. Linear baselines are often robust across sessions.
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=500, class_weight=None),
    "LinearSVC": LinearSVC(class_weight=None),  # we'll calibrate to get probs
    "GaussianNB": GaussianNB(),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

# Optional: try XGBoost/LightGBM if installed
TRY_XGBOOST = True
TRY_LIGHTGBM = False

In [ ]:
# =====================
# Imports / Dependencies
# =====================
import os, math, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, brier_score_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Imbalanced-learn (optional)
try:
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.over_sampling import RandomOverSampler, SMOTE
    HAVE_IMB = True
except Exception:
    HAVE_IMB = False
    warnings.warn("imblearn not available; 'undersample'/'oversample'/'smote' will be skipped.")

# Optional boosters
HAVE_XGB = False
if TRY_XGBOOST:
    try:
        from xgboost import XGBClassifier
        HAVE_XGB = True
    except Exception:
        pass

HAVE_LGBM = False
if TRY_LIGHTGBM:
    try:
        from lightgbm import LGBMClassifier
        HAVE_LGBM = True
    except Exception:
        pass

# Matplotlib defaults: single-plot rule respected; no explicit colors set.

In [ ]:
# =====================
# Utility Functions
# =====================

def ensure_output_dir(root: Path) -> Path:
    out = root / EXP_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    return out

def load_Xy(X_path: Path, y_path: Path, positive=POSITIVE_CLASS):
    X = pd.read_csv(X_path)
    y_df = pd.read_csv(y_path)
    if y_df.shape[1] == 1:
        y_raw = y_df.iloc[:,0].values
    else:
        # try a column named 'label'
        if "label" in y_df.columns:
            y_raw = y_df["label"].values
        else:
            raise ValueError("y.csv should have a single column or a 'label' column.")
    # Coerce labels to {0,1}
    if isinstance(positive, str):
        y = np.array([1 if str(v).strip().lower()==str(positive).lower() else 0 for v in y_raw])
    else:
        y = np.array([1 if v==positive else 0 for v in y_raw])
    return X, y

def align_columns(XA: pd.DataFrame, XB: pd.DataFrame):
    # Align feature sets to common columns and same order
    common = [c for c in XA.columns if c in set(XB.columns)]
    # Keep clip_id if present separately (for error analysis)
    meta_cols = [c for c in common if c.lower() in ("clip_id", "filename")]
    feat_cols = [c for c in common if c not in meta_cols]
    XA_aligned = XA[feat_cols + meta_cols].copy()
    XB_aligned = XB[feat_cols + meta_cols].copy()
    return XA_aligned, XB_aligned, feat_cols, meta_cols

def split_features_meta(X: pd.DataFrame, feat_cols, meta_cols):
    X_feats = X[feat_cols].values.astype(float)
    X_meta = X[meta_cols] if meta_cols else None
    return X_feats, X_meta

def class_balance(y):
    uniq, counts = np.unique(y, return_counts=True)
    return dict(zip(uniq, counts))

def describe_session(name, X, y):
    cb = class_balance(y)
    print(f"Session {name}: n={len(y)} | made={cb.get(1,0)} miss={cb.get(0,0)} (pos rate={cb.get(1,0)/max(1,len(y)):.2f})")
    if "clip_id" in X.columns:
        print(" - clip_id found; will include in error reports.")

def make_rebalancer(strategy, random_state=42):
    if strategy == "none":
        return None
    if strategy == "class_weight":
        return "class_weight"  # handled at model level when possible
    if not HAVE_IMB:
        print("imblearn not installed; skipping sampling strategy.")
        return None
    if strategy == "undersample":
        return RandomUnderSampler(random_state=random_state)
    if strategy == "oversample":
        return RandomOverSampler(random_state=random_state)
    if strategy == "smote":
        return SMOTE(random_state=random_state)
    return None

def wrap_model(name, base_est, calibrate=CALIBRATE, method=CALIB_METHOD, class_weight=False):
    est = base_est
    # Apply class_weight if requested and supported
    if class_weight and hasattr(est, "class_weight"):
        try:
            est.set_params(class_weight="balanced")
        except Exception:
            pass

    # LinearSVC doesn't give probs; wrap with calibration
    if name == "LinearSVC":
        est = CalibratedClassifierCV(estimator=est, cv=3, method=method)
        return Pipeline([("scaler", StandardScaler()), ("clf", est)])

    # Most other models benefit from scaling + optional calibration
    base_pipe = Pipeline([("scaler", StandardScaler()), ("clf", est)])

    if calibrate and hasattr(est, "predict_proba"):
        # We'll add a separate CalibratedClassifierCV wrapper after fitting if needed.
        # Simpler approach: rely on model's predict_proba + post-hoc CalibratedClassifierCV
        pass

    return base_pipe

def fit_calibrated(pipe, Xtr, ytr, method=CALIB_METHOD):
    try:
        clf = pipe.named_steps["clf"]
    except Exception:
        # unknown pipeline shape; just calibrate whole pipeline
        return CalibratedClassifierCV(base_estimator=pipe, cv=3, method=method).fit(Xtr, ytr)
    # If classifier supports predict_proba directly, calibrate on top of the pipeline
    try:
        _ = clf.predict_proba(np.zeros((1, Xtr.shape[1])))
        calibrated = CalibratedClassifierCV(base_estimator=pipe, cv=3, method=method).fit(Xtr, ytr)
        return calibrated
    except Exception:
        # Already calibrated (e.g., LinearSVC path) or no proba
        return pipe.fit(Xtr, ytr)

def predict_with_threshold(model, X, thr):
    # Try predict_proba; fallback to decision_function
    prob = None
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X)[:,1]
    else:
        try:
            df = model.decision_function(X)
            # Map decision scores to [0,1] via logistic proxy (monotonic); for ranking only
            prob = 1/(1+np.exp(-df))
        except Exception:
            prob = model.predict(X)  # last resort (hard labels)
    y_pred = (prob >= thr).astype(int)
    return y_pred, prob

def metrics_at_threshold(y_true, prob, thr):
    y_pred = (prob >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    p0, r0, f10, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    macro_f1 = np.mean([f10[0], f10[1]]) if isinstance(f10, np.ndarray) else np.nan
    ap = average_precision_score(y_true, prob)
    try:
        roc = roc_auc_score(y_true, prob)
    except Exception:
        roc = np.nan
    brier = brier_score_loss(y_true, prob)
    return dict(precision=p, recall=r, f1=f1, macro_f1=macro_f1, ap=ap, roc_auc=roc, brier=brier)

def sweep_thresholds(y_true, prob, objective="macro_f1", cost_fn=2.0, cost_fp=1.0):
    thr_vals = np.linspace(0.05, 0.95, 19)
    rows = []
    for t in thr_vals:
        m = metrics_at_threshold(y_true, prob, t)
        if objective == "macro_f1":
            score = m["macro_f1"]
        elif objective == "f1":
            score = m["f1"]
        elif objective == "custom":
            y_pred = (prob >= t).astype(int)
            # Custom cost
            fp = np.sum((y_true==0) & (y_pred==1))
            fn = np.sum((y_true==1) & (y_pred==0))
            score = -(cost_fn*fn + cost_fp*fp)  # max is best (less cost)
        else:
            score = m["macro_f1"]
        m["thr"] = t
        m["objective_score"] = score
        rows.append(m)
    df = pd.DataFrame(rows).sort_values("objective_score", ascending=False).reset_index(drop=True)
    return df

def plot_confusion(y_true, y_pred, title, out_path=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    fig = plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks([0,1],[0,1])
    plt.yticks([0,1],[0,1])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i,j]), ha="center", va="center")
    if out_path:
        fig.savefig(out_path, bbox_inches="tight")
    plt.show()

def plot_pr_curve(y_true, prob, title, out_path=None):
    from sklearn.metrics import precision_recall_curve
    p, r, thr = precision_recall_curve(y_true, prob)
    fig = plt.figure()
    plt.plot(r, p)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    if out_path:
        fig.savefig(out_path, bbox_inches="tight")
    plt.show()

def plot_reliability(y_true, prob, title, out_path=None):
    frac_pos, mean_pred = calibration_curve(y_true, prob, n_bins=10, strategy="uniform")
    fig = plt.figure()
    plt.plot([0,1],[0,1], linestyle="--")
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(title)
    if out_path:
        fig.savefig(out_path, bbox_inches="tight")
    plt.show()

def error_report(X_meta, y_true, prob, thr, out_csv):
    if X_meta is None or "clip_id" not in X_meta.columns:
        return
    y_pred = (prob >= thr).astype(int)
    df = X_meta.copy()
    df["y_true"] = y_true
    df["prob_make"] = prob
    df["y_pred"] = y_pred
    df["error_type"] = np.where((df["y_true"]==1)&(df["y_pred"]==0),"FN",
                         np.where((df["y_true"]==0)&(df["y_pred"]==1),"FP","OK"))
    df.to_csv(out_csv, index=False)
    print(f"Saved error report to {out_csv}")

In [ ]:
# ==============================
# Cross‑session Train → Test runs
# ==============================

results = []

# Prepare outputs
for s_name, (_, _, out_root) in SESSIONS.items():
    ensure_output_dir(out_root)

# Load both sessions
session_data = {}
for s_name, (Xp, yp, out_root) in SESSIONS.items():
    X, y = load_Xy(Xp, yp, POSITIVE_CLASS)
    describe_session(s_name, X, y)
    session_data[s_name] = (X, y, out_root)

# Align feature columns between the two sessions (ignores clip_id/filename in features)
names = list(session_data.keys())
(XA, yA, outA) = session_data[names[0]]
(XB, yB, outB) = session_data[names[1]]

XA_aligned, XB_aligned, feat_cols, meta_cols = align_columns(XA, XB)

print(f"Common feature count: {len(feat_cols)}")
if meta_cols:
    print(f"Meta columns: {meta_cols}")

# Build rebalancer
rebalancer = make_rebalancer(REBALANCE)

# Optionally extend models with boosters
if HAVE_XGB:
    MODELS["XGBClassifier"] = XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.06, subsample=0.9,
        colsample_bytree=0.8, reg_lambda=1.0, random_state=42, eval_metric="logloss"
    )
if HAVE_LGBM:
    MODELS["LGBMClassifier"] = LGBMClassifier(
        n_estimators=400, learning_rate=0.06, subsample=0.9,
        colsample_bytree=0.8, random_state=42
    )

def train_eval(train_name, test_name):
    Xtr_df, ytr, out_root_tr = session_data[train_name]
    Xte_df, yte, out_root_te = session_data[test_name]

    # align both to common feature set
    Xtr_df, Xte_df, feat_cols2, meta_cols2 = align_columns(Xtr_df, Xte_df)
    Xtr, Xtr_meta = split_features_meta(Xtr_df, feat_cols2, meta_cols2)
    Xte, Xte_meta = split_features_meta(Xte_df, feat_cols2, meta_cols2)

    for mname, m in MODELS.items():
        # class_weight path?
        want_class_weight = (REBALANCE == "class_weight")
        pipe = wrap_model(mname, m, calibrate=CALIBRATE, method=CALIB_METHOD, class_weight=want_class_weight)

        Xtrain_fit, ytrain_fit = Xtr, ytr
        # Sampling strategies applied only to training data
        if rebalancer not in (None, "class_weight"):
            # Apply after scaling? For simplicity, apply in raw space here.
            Xtrain_fit, ytrain_fit = rebalancer.fit_resample(Xtr, ytr)

        model = fit_calibrated(pipe, Xtrain_fit, ytrain_fit, method=CALIB_METHOD)

        # Probabilities on test
        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(Xte)[:,1]
        else:
            try:
                df = model.decision_function(Xte)
                prob = 1/(1+np.exp(-df))
            except Exception:
                prob = model.predict(Xte).astype(float)

        # Threshold sweep
        sweep = sweep_thresholds(yte, prob, THRESHOLD_OBJECTIVE, COST_FN, COST_FP)
        best = sweep.iloc[0].to_dict()
        thr = float(best["thr"])

        # Final predictions
        ypred = (prob >= thr).astype(int)

        # Metrics
        report = {
            "train_session": train_name,
            "test_session": test_name,
            "model": mname,
            "rebalance": REBALANCE,
            "calibrated": CALIBRATE,
            "thr": thr,
            **{k: float(best[k]) for k in ["precision","recall","f1","macro_f1","ap","roc_auc","brier"]}
        }
        results.append(report)

        # Plots & reports
        out_dir = ensure_output_dir(out_root_te)
        base = f"{train_name}_to_{test_name}__{mname}_{REBALANCE}"
        plot_confusion(yte, ypred, f"CM {base} (thr={thr:.2f})", out_dir / f"cm_{base}.png")
        plot_pr_curve(yte, prob, f"PR {base}", out_dir / f"pr_{base}.png")
        plot_reliability(yte, prob, f"Reliability {base}", out_dir / f"reliability_{base}.png")
        error_report(Xte_meta, yte, prob, thr, out_dir / f"errors_{base}.csv")

    return pd.DataFrame(results)

df_res_1 = train_eval(names[0], names[1])
df_res_2 = train_eval(names[1], names[0])

all_results = pd.DataFrame(results).sort_values(["macro_f1","f1","ap"], ascending=False)
display(all_results)

In [ ]:
# ============================
# GroupKFold (by session) sweep
# ============================

# Combine sessions for group-aware selection
# Build a joint X/y with a 'group' array for session_id
joint_X_list, joint_y_list, joint_groups_list = [], [], []
session_idx = {name:i for i,name in enumerate(SESSIONS.keys())}

for s_name, (Xp, yp, _) in SESSIONS.items():
    Xs, ys = load_Xy(Xp, yp, POSITIVE_CLASS)
    Xs = Xs[[c for c in Xs.columns if c not in ("clip_id","filename")]]
    joint_X_list.append(Xs)
    joint_y_list.append(ys)
    joint_groups_list.append(np.full(len(ys), session_idx[s_name], dtype=int))

X_joint = pd.concat(joint_X_list, axis=0, ignore_index=True)
y_joint = np.concatenate(joint_y_list, axis=0)
groups_joint = np.concatenate(joint_groups_list, axis=0)

# Rebalancer setup (applied within each fold on train indices)
rebalancer = make_rebalancer(REBALANCE)

def group_cv_eval():
    rows = []
    gkf = GroupKFold(n_splits=len(SESSIONS))
    for mname, m in MODELS.items():
        want_class_weight = (REBALANCE == "class_weight")
        pipe = wrap_model(mname, m, calibrate=CALIBRATE, method=CALIB_METHOD, class_weight=want_class_weight)
        fold_metrics = []
        for fold, (tr, te) in enumerate(gkf.split(X_joint, y_joint, groups_joint)):
            Xtr = X_joint.iloc[tr].values.astype(float)
            ytr = y_joint[tr]
            Xte = X_joint.iloc[te].values.astype(float)
            yte = y_joint[te]

            Xtr_fit, ytr_fit = Xtr, ytr
            if rebalancer not in (None, "class_weight"):
                Xtr_fit, ytr_fit = rebalancer.fit_resample(Xtr, ytr)

            model = fit_calibrated(pipe, Xtr_fit, ytr_fit, method=CALIB_METHOD)

            if hasattr(model, "predict_proba"):
                prob = model.predict_proba(Xte)[:,1]
            else:
                try:
                    df = model.decision_function(Xte)
                    prob = 1/(1+np.exp(-df))
                except Exception:
                    prob = model.predict(Xte).astype(float)

            sweep = sweep_thresholds(yte, prob, THRESHOLD_OBJECTIVE, COST_FN, COST_FP)
            best_thr = float(sweep.iloc[0]["thr"])
            ypred = (prob >= best_thr).astype(int)
            metr = metrics_at_threshold(yte, prob, best_thr)
            fold_metrics.append(metr)

        # Aggregate
        agg = {k: float(np.mean([m[k] for m in fold_metrics])) for k in fold_metrics[0].keys()}
        rows.append({"model": mname, "rebalance": REBALANCE, "calibrated": CALIBRATE, **agg})
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False)

group_cv_results = group_cv_eval()
display(group_cv_results)

In [ ]:
# =================
# Save summary CSVs
# =================
for s_name, (_, _, out_root) in SESSIONS.items():
    out_dir = ensure_output_dir(out_root)
    if 'all_results' in globals():
        all_results.to_csv(out_dir / "cross_session_results.csv", index=False)
    if 'group_cv_results' in globals():
        group_cv_results.to_csv(out_dir / "groupcv_results.csv", index=False)

print("Saved results to each session's experiments folder.")